# 🎬 Manhwa Video Studio — Google Colab Edition

Run **Manhwa Video Studio** in Google Colab's cloud environment with fast cloud vCPUs, pre-configured FFmpeg, and an instant public HTTPS link.

### ✨ Why Run on Colab?
- 🚀 **Zero Local Resource Usage**: Renders 1080p/2K videos without heating up your computer or consuming your RAM.
- ⚡ **Datacenter Network Speed**: Downloads manhwa chapter images in seconds directly from CDNs.
- 📱 **Access Anywhere**: Open the studio on your phone, iPad, or laptop via a secure HTTPS link.
- 💯 **100% Free**: Uses Google Colab's free tier with no account needed for the Cloudflare tunnel.

## 📦 Step 1: Install Dependencies
Install FFmpeg and required Python packages.

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q fastapi "uvicorn[standard]" requests pillow numpy pydantic python-multipart

## 📁 Step 2: Get the Project Files
Choose **Option A** (if cloning from a Git repository) or **Option B** (if uploading a ZIP of your project).

In [ ]:
import os, zipfile

# Check if app.py is already present
if not os.path.exists('app.py'):
    print("Choose one of the methods below to load your code:")
    # OPTION A: If your code is on GitHub, uncomment and edit the line below:
    # !git clone https://github.com/YOUR_USERNAME/manhwa-video-studio.git && cp -r manhwa-video-studio/* ./
    
    # OPTION B: Upload your project ZIP file from your computer:
    from google.colab import files
    print("Please upload your 'manhwa-video-studio.zip' archive:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            with zipfile.ZipFile(fn, 'r') as zip_ref:
                zip_ref.extractall('.')
            print(f"Extracted {fn} successfully!")
else:
    print("✅ Project files (app.py, static/) already present in workspace!")

## 🚀 Step 3: Launch Studio with Free Cloudflare Tunnel (1-Click)
Runs the FastAPI server and creates a secure public HTTPS link (`https://xxxx.trycloudflare.com`). **No signup or token required!**

In [ ]:
import subprocess
import time
import os
import re
from IPython.display import display, HTML

# 1. Download Cloudflare Tunnel binary (free, no account needed)
if not os.path.exists("cloudflared"):
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

# 2. Kill any previous instance
!pkill -f "uvicorn app:app" || true
!pkill -f "cloudflared" || true
time.sleep(1)

# 3. Start Uvicorn backend in background
server_proc = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(2)

# 4. Launch Cloudflare Tunnel
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

print("Starting tunnel... finding public URL...")
public_url = None
start = time.time()
while time.time() - start < 35:
    line = tunnel_proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("=" * 60)
    print(f"🎉 STUDIO ONLINE: {public_url}")
    print("=" * 60)
    display(HTML(f'''
    <div style="background: #111827; border: 2px solid #f59e0b; padding: 20px; border-radius: 12px; margin: 15px 0;">
      <h2 style="color: #fbbf24; margin-top: 0;">🎬 Manhwa Video Studio is Live!</h2>
      <p style="color: #d1d5db; font-size: 14px;">Click the button below to open the studio in a new tab:</p>
      <a href="{public_url}" target="_blank" style="background: linear-gradient(135deg, #f59e0b, #d97706); color: #000; padding: 12px 24px; border-radius: 8px; font-weight: bold; text-decoration: none; display: inline-block; font-size: 16px;">
        🚀 Open Manhwa Video Studio
      </a>
    </div>
    '''))
else:
    print("Could not automatically detect the URL. Run '!pkill -f cloudflared' and retry.")

## 🌐 Alternative: LocalTunnel (Fallback)
If Cloudflare is not available in your region, run this cell for a LocalTunnel URL instead:

In [ ]:
# Get external IP for LocalTunnel password bypass
!curl -s ipv4.icanhazip.com
print("Copy the IP address above as your password if prompted by LocalTunnel!")
!npx -y localtunnel --port 8000